In [ ]:
%load_ext autoreload
%autoreload 2
    
import os
import pandas as pd
import math
import matplotlib.pyplot as plt
import numpy as np
import random

from scipy.special import erfc, gamma, gammaincc
from scipy.integrate import quad
import itertools


In [ ]:
def getFalseHitProb(threshold, ENC):
    p = 0.5*erfc(threshold/(ENC*math.sqrt(2)))
    return p

In [ ]:
print(getFalseHitProb(10,3))

In [ ]:
# Make figure and add plots
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

len = 3
threshold = np.logspace(0, len, 100*len+1)
ENC = [3, 10, 50, 100]

for inENC in ENC:
    p = getFalseHitProb(threshold, inENC)
    ax[0].plot(
        threshold, p,
        label=f'ENC = {inENC}'
    )
    ax[1].plot(
        threshold/inENC, p,
        label=f'ENC = {inENC}'
    )

ax[0].set_xlabel(r'Threshold: $n_\text{th}$')
ax[1].set_xlabel(r'Threshold Factor: $n_\text{th}$ / ENC')

for inAx in ax:
    #inAx.set_xscale('log')
    inAx.set_ylabel('False Hit Probability')
    inAx.grid()
ax[0].legend()
ax[1].set_yscale('log')
ax[1].set_xlim([.9, 11])
ax[1].set_ylim([1e-21, 2])

plt.tight_layout()
plt.show()

In [ ]:
print(getFalseHitProb(24/3))

In [ ]:
def myPolya(n, gain, theta):
    A = 1/gain
    B = np.power(theta+1, theta+1)
    C = 1/gamma(theta+1)
    D = np.power(n/gain, theta)
    E = np.exp(-n/gain*(theta+1))

    result = A*B*C*D*E
    return result

def signalAboveThreshold(x, threshold, gain, theta, ENC):

    probAvalancheSize = myPolya(x, gain, theta)
    num = threshold - x
    denom = np.sqrt(2)*ENC
    probAboveThreshold = 0.5*erfc(num/denom)

    result = probAvalancheSize*probAboveThreshold

    return result

def efficiencyAtThreshold(threshold, gain, theta, ENC):
    efficiency, _ = quad(
        signalAboveThreshold,
        a=0.0,
        b=10*gain,
        args=(threshold, gain, theta, ENC)
    )

    return efficiency

In [ ]:
gain = 50
theta = 1.5
ENC = 3
threshold =10


eff = efficiencyAtThreshold(threshold, gain, theta, ENC)
print(eff)

In [ ]:
thetas = [0, 1, 2]
thresholds = np.linspace(0, gain*2.5, 150)
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

axs[0].axvline(
    gain, 
    c='m', ls='--', lw=2, label=f'Gain = {gain}'
)

for theta in thetas:
    efficiencies = [
        efficiencyAtThreshold(th, gain, theta, ENC) 
        for th in thresholds
    ]
    
    charge = np.linspace(0, gain*2.5, 500)
    polyaProb = myPolya(charge, gain, theta)
    
    
    
    axs[0].plot(
        charge, polyaProb, 
        label=rf'Polya ($\theta$={theta:.1f})'
    )
    '''
    axs[0].fill_between(
        charge, polyaProb, alpha=0.5, color='m'
    )
    
    axs[0].axvline(
        10, 
        c='r', ls='--', lw=2, label=f'Threshold = 10e'
    )

    '''
    
    
    
    
    
    axs[1].plot(
        thresholds, efficiencies, 
        label=rf'S-Curve ($\theta$={theta:.1f})'
    )
'''
axs[1].axvline(
    10, 
    c='r', ls='--', lw=2, label=f'Threshold = 10e'
)'''

axs[0].set_xlabel('Avalanche Size: n')
axs[0].set_ylabel(r'PDF: P(n)')

axs[1].set_xlabel('Threshold (e)')
axs[1].set_ylabel('Detection Efficiency (%)')

for inAx in axs:
    inAx.grid()
    inAx.legend()

plt.tight_layout()
plt.show()

In [ ]:
length = 3
thetas = [0, 1, 2]
thresholds = np.linspace(1, gain*length, 100)
charge = np.linspace(0, gain*length, 101)
fig, axs = plt.subplots(1, 3, figsize=(12, 4))

axs[0].axvline(
    gain/gain, 
    c='m', ls='--', lw=2, label=f'Mean Gain'
)

for theta in thetas:
    efficiencies = [
        efficiencyAtThreshold(th, gain, theta, ENC) 
        for th in thresholds
    ]
    
    
    polyaProb = myPolya(charge, gain, theta)
    
    
    
    axs[0].plot(
        charge/gain, polyaProb*gain, 
        label=rf'Polya ($\theta$={theta:.1f})'
    )
    '''
    axs[0].fill_between(
        charge, polyaProb, alpha=0.5, color='m'
    )
    
    axs[0].axvline(
        10, 
        c='r', ls='--', lw=2, label=f'Threshold = 10e'
    )

    '''
    
    
    
    
    
    axs[1].plot(
        thresholds/gain, efficiencies, 
        label=rf'S-Curve ($\theta$={theta:.1f})'
    )
    axs[2].plot(
        gain/thresholds, efficiencies, 
        label=rf'S-Curve ($\theta$={theta:.1f})'
    )
'''
axs[1].axvline(
    10, 
    c='r', ls='--', lw=2, label=f'Threshold = 10e'
)'''

axs[0].set_xlabel(r'Avalanche Size: n / $\bar{n}$')
axs[0].set_ylabel(r'PDF: $\bar{n}$ * P(n)')

axs[1].set_xlabel('Threshold / Gain : 1/GTR')
axs[1].set_ylabel('Detection Efficiency (%)')

axs[2].set_xlabel('Gain / Threshold: GTR')
axs[2].set_ylabel('Detection Efficiency (%)')

axs[2].set_xscale('log')

for inAx in axs:
    inAx.grid()
    inAx.legend()

plt.tight_layout()
plt.show()

In [ ]:
def myPolyaNormal(nNormal, theta):
    A = np.power(theta+1, theta+1)
    B = 1/gamma(theta+1)
    C = np.power(nNormal, theta)
    D = np.exp(-nNormal*(theta+1))
    polyaPDF = A*B*C*D
    return polyaPDF

def myPolyaEfficiencyNormal(GTR, theta):    
    s = theta+1
    x = s/GTR
    efficiency = gammaincc(s, x)
    return efficiency

def riceNoiseRate(TNR, freq):
    riceRate = freq*np.exp(-0.5*TNR**2)
    return riceRate

def riceOccupancy(TNR, freq, dt):
    riceRate = riceNoiseRate(TNR, freq)
    probability = -np.expm1(-riceRate*dt)
    return np.clip(probability, 0.0, 1.0)

In [ ]:
thetas = np.array([0, 1, 2])

freq = 100e6 # 100 MHz
dt = 10e-9 #10 ns

n = np.linspace(0, 5, 501)
GTR = np.logspace(-1, 2, 301)
TNR = np.linspace(0.5, 7.0, 300)

fig, axs = plt.subplots(2, 2, figsize=(12, 8))
axs = axs.flatten()

# Signal Efficiencies
for th in thetas:

    polyaShape = myPolyaNormal(n, th)
    axs[0].plot(
        n, polyaShape,
        label=rf'$\theta$={th:.1f}'
    )

    polyaEff = myPolyaEfficiencyNormal(GTR, th)
    axs[1].plot(
        GTR, polyaEff,
        label=rf'$\theta$={th:.1f}'
    )    
    axs[2].plot(
        1/GTR, polyaEff,
        label=rf'$\theta$={th:.1f}'
    )

# Noise Occupancies
axs[3].plot(
    TNR, riceOccupancy(TNR, freq, dt),
    c='r', lw=2,
    label=rf'f={freq/1e6:.0f} MHz, dt={dt*1e9:.0f} ns'
)

for ax in axs:
    ax.grid()
    ax.legend()

axs[1].set_xscale('log')
axs[2].set_xscale('log')
axs[3].set_xscale('log')

axs[0].set_xlabel(r'Normalized Gain: $\bar{n} / n$')
axs[1].set_xlabel('Gain to Threshold Ratio: GTR')
axs[2].set_xlabel('Threshold / Gain: 1/GTR')
axs[3].set_xlabel('Threshold / Noise: TNR')

axs[0].set_ylabel(r'Polya PDF: $\bar{n} * P(n)$')
axs[1].set_ylabel('Detection Efficiency')
axs[2].set_ylabel('Detection Efficiency')
axs[3].set_ylabel('Noise Occupancy')


plt.tight_layout()
plt.show()

In [ ]:
# Setup
freq = 1e8 
dt = 10e-9

GTR = np.logspace(-1, 2, 301)
TNR = np.linspace(0, 10, 301)

ployaEff_t1 = myPolyaEfficiencyNormal(GTR, 1)
ployaEff_t2 = myPolyaEfficiencyNormal(GTR, 2)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))

for ax in axs:
    
    ax.fill_between(
        1/GTR, ployaEff_t1, ployaEff_t2,
        color='r', alpha=0.5,
        label=r'Polya Efficiency ($1\leq\theta\leq2$)'
    )
    
    ax.plot(
        TNR, riceOccupancy(TNR, freq, dt),
        c='b', lw=2, label=f'Rice Noise Occupancy (f={freq/1e6:.0f} MHz, dt={dt/1e-9:.0f} ns)'
    )

    ax.set_xlabel('Threshold/Gain (1/GTR) and Threshold/Noise (TNR)')
    
    ax.set_ylabel('Signal Efficiency')
    ax.grid()
    ax.legend()

axs[1].set_xscale('log')

plt.tight_layout()
plt.show()

In [ ]:
# Setup
freq = 1e8 
dt = 10e-9

GTR = np.logspace(-1, 2, 301)
TNR = np.linspace(0.01, 10, 301)

ployaEff_t1 = myPolyaEfficiencyNormal(GTR, 1)
ployaEff_t2 = myPolyaEfficiencyNormal(GTR, 2)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))

for ax in axs:
    
    ax.fill_between(
        GTR, ployaEff_t1, ployaEff_t2,
        color='r', alpha=0.5,
        label=r'Polya Efficiency ($1\leq\theta\leq2$)'
    )
    
    ax.plot(
        1/TNR, riceOccupancy(TNR, freq, dt),
        c='b', lw=2, label=f'Rice Noise Occupancy (f={freq/1e6:.0f} MHz, dt={dt/1e-9:.0f} ns)'
    )

    ax.set_xlabel('Gain/Threshold (GTR) and Noise/Threshold (1/TNR)')
    
    ax.set_ylabel('Signal Efficiency')
    ax.grid()
    ax.legend()

axs[1].set_xscale('log')

plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

ENC = np.array([3, 60])
TNR = 5.0

gain = np.logspace(0, 4, 500)
for ax in axs:
    for noise in ENC:
        threshold = noise*TNR
        
        ax.fill_between(
            gain, 
            myPolyaEfficiencyNormal(gain/threshold, 1), 
            myPolyaEfficiencyNormal(gain/threshold, 2),
            label=rf'Polya Efficiency ($1\leq\theta\leq2$), ENC = {noise}'
        )

    ax.grid()
    ax.legend()
    ax.set_xlabel(r'Gain: $\bar{n}$')
    ax.set_ylabel('Detection Efficiency')

axs[1].set_xscale('log')
plt.tight_layout()

In [ ]:
# Setup
freq = 100e6
dt = 100e-9

ENC = np.array([3, 60])
gains = [1e2, 1e3]

# Sweep Threshold in Noise Sigmas (TNR = Q_th / ENC)
TNR = np.logspace(-1, 3, 501)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))

for ax, gain in zip(axs, gains):
    ax.plot(
        TNR, 
        riceOccupancy(TNR, freq, dt),
        color='r', ls='--', lw=2.5,
        label=f'Rice Noise Occupancy (f={freq/1e6:.0f} MHz, dt={dt/1e-9:.0f} ns)'
    )

    for noise in ENC:
        GTR = gain/(TNR*noise)
        ax.fill_between(
            TNR, 
            myPolyaEfficiencyNormal(GTR, 1), 
            myPolyaEfficiencyNormal(GTR, 2),
            label=rf'Polya Efficiency ($1\leq\theta\leq2$), ENC = {noise}'
        )

    ax.grid()
    ax.legend()
    ax.set_xscale('log')
    ax.set_xlabel('Threshold / Noise: (TNR)')
    ax.set_ylabel('Occupancy / Efficiency')
    ax.set_title(rf'Gas Gain: $\bar n $ = {gain}')

plt.tight_layout()
plt.show()